## 1. 依存パッケージ

ローカル環境に必要なパッケージが無い場合は、先頭の `#` を外して実行する。

In [1]:
 # !pip install -U pip
 # !pip install numpy gymnasium tensorboard torch "imageio[ffmpeg]" pillow
 # !pip install mujoco mujoco-warp warp-lang

## 2. インポートと実行環境の判定

ライブラリを読み込む。Google Colab で開かれた場合は Drive をマウントし、依存パッケージを自動インストールする。

In [2]:
import os
# ヘッドレス環境でのオフスクリーン描画 (mujoco.Renderer) に EGL を使う
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import sys
ENV_COLAB = "google.colab" in sys.modules
if ENV_COLAB:
    from google.colab import drive, runtime
    drive.mount('/content/drive')
    %cd /content/drive/My Drive/Colab Notebooks
    !pip install gymnasium tensorboard mujoco mujoco-warp warp-lang "imageio[ffmpeg]" pillow

import atexit
from copy import deepcopy
from datetime import datetime
from importlib import metadata
import itertools
import shutil
import tempfile
import time
import traceback
import warnings

import imageio
import numpy as np
from PIL import Image, ImageDraw, ImageFont

import mujoco
import mujoco_warp as mjw
import warp as wp
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

wp.config.log_level = 30          # warp の起動バナーを抑制
wp.init()

mjw_ver = getattr(mjw, "__version__", None) or metadata.version("mujoco-warp")
print(f"mujoco {mujoco.__version__} / mujoco_warp {mjw_ver} / warp {wp.__version__} "
      f"/ torch {torch.__version__}")


mujoco 3.13.0 / mujoco_warp 3.13.0 / warp 1.17.0 / torch 2.5.1+cu121


## 3. 参照モーション生成器(逆運動学トロット)

学習のお手本になる関節角軌道をつくるセル。対角2脚ずつ足を運ぶトロットの足先軌道を計画し、脚の逆運動学で 12 関節の角度テーブル(1周期分)に変換する。

このセルに書かれているもの:

- `leg_fk()` — 関節角 → 足先位置の順運動学(逆運動学の検証用)
- `leg_ik()` — 足先位置 → 3関節角の解析逆運動学
- `standing_pose()` — 指定した胴体高さで立つ立位関節角
- `OmniTrot` — 指令速度 (vx, vy, ωz) からトロット1周期の関節角テーブルを作る参照モーション生成器
- `check_limits()` — 参照モーションが関節可動域・IK 到達範囲に収まるかの一括チェック

In [3]:
"""Go2 の全方向トロット参照モーション生成器。

速度指令 (vx, vy, ωz) に対し、足の接地位置を決めてから逆運動学で関節角を求める。

- 前進/後退/横移動/旋回とその組み合わせを同じ枠組みで生成する
- 横移動・旋回には外転(hip)関節が要るため、IK は 3 自由度の解析解
- 接地脚の足先はワールド座標に固定される (構造的に滑らない)
"""
import numpy as np

# ---- go2.xml から読み取った寸法 ----
L1     = 0.213                                # thigh リンク長
L2     = float(np.hypot(0.002, 0.213))        # calf 等価リンク長 (足geomは calf 座標系で (-0.002, 0, -0.213))
DELTA  = float(np.arctan2(0.002, 0.213))      # calf リンクの前後オフセット角
D_HIP  = 0.0955                               # 外転ピボット→thigh関節の横オフセット
FOOT_R = 0.022                                # 足geom半径 = 接地時の足中心高さ

ABD_RANGE  = (-1.0472, 1.0472)
CALF_RANGE = (-2.7227, -0.83776)

# (名前, hip_x, hip_y, 左右符号s, 位相オフセット, thigh可動域)
# トロット: 対角ペアが同位相 FL,RR = 0.0 / FR,RL = 0.5
LEGS = [
    ("FL",  0.1934,  0.0465, +1, 0.0, (-1.5708, 3.4907)),
    ("FR",  0.1934, -0.0465, -1, 0.5, (-1.5708, 3.4907)),
    ("RL", -0.1934,  0.0465, +1, 0.5, (-0.5236, 4.5379)),
    ("RR", -0.1934, -0.0465, -1, 0.0, (-0.5236, 4.5379)),
]
QPOS_ORDER = ["FL", "FR", "RL", "RR"]         # go2.xml の qpos[7:] の脚の並び


# ---------------------------------------------------------------- 脚のFK/IK

def _rx(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def _ry(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def leg_fk(angles, s):
    """(θ_abd, θ_thigh, θ_calf) → 外転ピボット基準の足中心位置 (x,y,z)。"""
    t1, t2, t3 = angles
    p = (np.array([0.0, s * D_HIP, 0.0])
         + _ry(t2) @ np.array([0.0, 0.0, -L1])
         + _ry(t2 + t3) @ np.array([-0.002, 0.0, -0.213]))
    return _rx(t1) @ p


def leg_ik(p, s):
    """外転ピボット基準の足先目標 p=(x,y,z) → (θ_abd, θ_thigh, θ_calf)。解析解。

    1) 外転角: 脚の矢状面は外転軸から距離 |D_HIP| の平面にあるので、
       y-z 平面で cosθ1·py + sinθ1·pz = s·D_HIP を解く (足が下にある分岐を選択)。
    2) 残りは矢状面内の 2 リンク IK (余弦定理)。膝の可動域が負なので分岐は一意。
    """
    px, py, pz = p
    d = s * D_HIP
    r = np.hypot(py, pz)
    th1 = np.arctan2(pz, py) + np.arccos(np.clip(d / r, -1.0, 1.0))
    qz = -np.sqrt(max(r * r - d * d, 1e-12))          # 矢状面内での足の深さ
    c3 = np.clip((px * px + qz * qz - L1 * L1 - L2 * L2) / (2 * L1 * L2), -1.0, 1.0)
    ph3 = -np.arccos(c3)                               # 膝は常に負方向(後方)に畳む
    th2 = np.arctan2(-px, -qz) - np.arctan2(L2 * np.sin(ph3), L1 + L2 * np.cos(ph3))
    th3 = ph3 - DELTA
    return np.array([th1, th2, th3])


def standing_pose(h0=0.33):
    """高さ h0 で足を公称接地点に置いた立位の関節角12個 (qpos[7:] の並び)。"""
    q = np.zeros(12)
    for i, (name, hx, hy, s, off, trng) in enumerate(LEGS):
        local = np.array([0.0, s * D_HIP, FOOT_R - h0])
        k = QPOS_ORDER.index(name)
        q[3 * k:3 * k + 3] = leg_ik(local, s)
    return q


# ---------------------------------------------------------------- 生成器

class OmniTrot:
    """速度指令 (vx, vy, ωz) から全方向トロットの参照モーションを生成する。

    vx, vy : 胴体座標系の前後・左右速度 [m/s]
    wz     : ヨー角速度 [rad/s]
    T      : ストライド周期 [s] / beta: デューティ比 / h0: 胴体高さ [m]
    h_swing: 遊脚の持ち上げ高さ [m]

    接地脚の足先(foothold)はワールド座標に固定する。foothold は「接地中間時刻での
    公称接地点の位置」に置く (Raibert 風)。
    """

    def __init__(self, vx=0.0, vy=0.0, wz=0.0,
                 T=0.4, beta=0.5, h0=0.33, h_swing=0.05):
        self.vx, self.vy, self.wz = float(vx), float(vy), float(wz)
        self.T, self.beta, self.h0, self.h_swing = T, beta, h0, h_swing

    # -- 胴体のワールド軌道 (等速の並進+旋回 → 円弧) --
    def base_pose(self, t):
        """時刻 t の胴体位置 (x, y) とヨー角 ψ。"""
        w = self.wz
        ps = w * t
        if abs(w) < 1e-9:
            return self.vx * t, self.vy * t, ps
        x = (self.vx * np.sin(ps) + self.vy * (np.cos(ps) - 1.0)) / w
        y = (self.vx * (1.0 - np.cos(ps)) + self.vy * np.sin(ps)) / w
        return x, y, ps

    # -- foothold: 第nストライドの接地点 (ワールドxy) --
    def _foothold(self, i, n):
        name, hx, hy, s, off, trng = LEGS[i]
        t_mid = (n - off + 0.5 * self.beta) * self.T   # 接地区間の中間時刻
        x, y, ps = self.base_pose(t_mid)
        c, sn = np.cos(ps), np.sin(ps)
        nx, ny = hx, hy + s * D_HIP                    # 公称接地点(胴体座標)
        return np.array([x + c * nx - sn * ny, y + sn * nx + c * ny])

    # -- 足先のワールド位置 --
    def foot_world(self, i, t):
        """時刻 t の脚 i の足中心ワールド位置。接地中は foothold に固定し、遊脚中は次の foothold へ補間する。"""
        off = LEGS[i][4]
        u = t / self.T + off
        n = int(np.floor(u))
        ph = u - n
        if ph < self.beta:                             # 接地: ワールドに固定
            f = self._foothold(i, n)
            return np.array([f[0], f[1], FOOT_R])
        sw = (ph - self.beta) / (1.0 - self.beta)      # 遊脚: 次のfootholdへ
        f0, f1 = self._foothold(i, n), self._foothold(i, n + 1)
        xy = f0 + (f1 - f0) * 0.5 * (1.0 - np.cos(np.pi * sw))
        z = FOOT_R + self.h_swing * np.sin(np.pi * sw)
        return np.array([xy[0], xy[1], z])

    # -- 参照 qpos (19) --
    def qpos_at(self, t):
        """時刻 t の参照 qpos (19): 胴体位置3 + 姿勢クォータニオン4 + 関節12。"""
        x, y, ps = self.base_pose(t)
        c, sn = np.cos(ps), np.sin(ps)
        q = np.zeros(19)
        q[0:3] = [x, y, self.h0]
        q[3], q[6] = np.cos(ps / 2), np.sin(ps / 2)    # ヨーのみのクォータニオン
        for i, (name, hx, hy, s, off, trng) in enumerate(LEGS):
            hipw = np.array([x + c * hx - sn * hy, y + sn * hx + c * hy, self.h0])
            d = self.foot_world(i, t) - hipw
            local = np.array([c * d[0] + sn * d[1],    # ワールド→胴体(ヨー)座標
                              -sn * d[0] + c * d[1], d[2]])
            k = QPOS_ORDER.index(name)
            q[7 + 3 * k:10 + 3 * k] = leg_ik(local, s)
        return q

    # -- 参照 qvel (18): ベースは解析、関節は有限差分 --
    def qvel_at(self, t, eps=1e-4):
        ps = self.wz * t
        c, sn = np.cos(ps), np.sin(ps)
        v = np.zeros(18)
        v[0] = c * self.vx - sn * self.vy              # ワールド系の線速度
        v[1] = sn * self.vx + c * self.vy
        v[5] = self.wz                                 # ヨーのみなので局所系でも同じ
        q0, q1 = self.qpos_at(t - eps), self.qpos_at(t + eps)
        v[6:] = (q1[7:] - q0[7:]) / (2 * eps)
        return v

    # -- 1周期ぶんの関節参照 (模倣学習の報酬テーブル用) --
    def joint_cycle(self, fps=50):
        """1ストライド周期の関節角・関節速度テーブル (n,12)×2。T*fps は整数にすること。"""
        n = int(round(self.T * fps))
        q = np.array([self.qpos_at(k / fps)[7:] for k in range(n)])
        dq = np.array([self.qvel_at(k / fps)[6:] for k in range(n)])
        return q, dq

    # -- 任意長のモーション --
    def qpos_motion(self, duration, fps=50):
        n = int(round(duration * fps))
        return np.array([self.qpos_at(k / fps) for k in range(n)])


def check_limits(motion):
    """モーション(…,19)の12関節が可動域内かを返す (ok, 詳細リスト)。"""
    rngs = []
    for k, name in enumerate(QPOS_ORDER):
        i = [l[0] for l in LEGS].index(name)
        trng = LEGS[i][5]
        rngs += [ABD_RANGE, trng, CALF_RANGE]
    ok, rows = True, []
    q = motion.reshape(-1, motion.shape[-1])
    for j, (lo, hi) in enumerate(rngs):
        mn, mx = q[:, 7 + j].min(), q[:, 7 + j].max()
        inside = (lo - 1e-9 <= mn) and (mx <= hi + 1e-9)
        ok &= inside
        rows.append((mn, mx, lo, hi, inside))
    return ok, rows



## 4. ハイパーパラメータ

このノートで使う設定をまとめて定義する: 歩容(周期 `T_GAIT`・胴体高さ `H0`・遊脚高さ `H_SWING`・指令レンジ `CMD_SCALE`)、サーボ PD ゲイン(`SERVO_KP/KD`)と行動スケール(`ACT_SCALE`)、TD3 の学習設定(学習率・バッチサイズ・総ステップ数など)、モデル・動画・ログの保存先。環境変数 `GO2_SMOKE=1` で起動すると動作確認用の短縮設定になる。

In [4]:
# ---- 歩容と指令レンジ ----
T_GAIT, H0, H_SWING = 0.40, 0.33, 0.05     # ストライド周期 [s] / 胴体高さ [m] / 遊脚高さ [m]
CMD_SCALE = np.array([0.60, 0.30, 1.00])   # |vx|max [m/s], |vy|max [m/s], |ωz|max [rad/s]
CYC = int(round(T_GAIT / 0.02))            # 1歩容周期の制御ステップ数 = 20

# ---- サーボ(位置)制御 ----
SERVO_KP, SERVO_KD = 60.0, 2.0             # 低レベル PD ゲイン (500 Hz)
# ACT_SCALE = np.asarray([0.395, 0.395, 0.757] * 4, dtype=np.float64)   
ACT_SCALE = np.asarray([0.5, 0.8, 0.8] * 4, dtype=np.float64)   # 行動 → 目標角の振幅 [rad]

# ---- GPU バッチ物理 ----
NUM_ENVS = 256               # 同時に進めるワールド数
PHYSICS_PRESET = "balanced"  # "exact" (go2.xml の option のまま) / "balanced" / "fast"
USE_CUDA_GRAPH = True        # 物理 10 刻み + PD を CUDA グラフに固める
                             # (ドライバの CUDA が 12.3 未満なら False)

# ---- 評価動画 ----
# 1 タスクぶんの解像度。動画は 2x2 なのでこの 2 倍の大きさになる
VIDEO_PANEL = (320, 240) if ENV_COLAB else (640, 480)

# ---- SNN ----
SNN_COMPILE = True           # actor と損失関数を torch.compile する (初回に 1〜2 分)

# ---- ハイパーパラメータ ----
# GO2_SMOKE=1 で動作確認用の縮小構成になる (次元と反復数のみ縮小)
SMOKE_TEST = os.environ.get("GO2_SMOKE", "0") == "1"
gpu_id = 0
if gpu_id >= torch.cuda.device_count():      # GPU が 1 枚の環境では 0 を使う
    gpu_id = 0
if SMOKE_TEST:
    NUM_ENVS = 64
    total_iters, eval_interval = 40, 20
    start_iters, update_after_iters, updates_per_iter = 3, 5, 2
    batch_size, max_ep_len = 32, 100
    replay_size = 20_000
    encoder_pop_dim = decoder_pop_dim = 64
    hidden = (128, 128)
    save_interval = 20
else:
    # 1 反復 = NUM_ENVS 本の遷移 + updates_per_iter 回の勾配更新
    total_iters, eval_interval = 10_000, 1_000
    start_iters, update_after_iters, updates_per_iter = 20, 40, 8
    batch_size, max_ep_len = 256, 500
    replay_size = 100_000
    encoder_pop_dim, decoder_pop_dim = 64, 256
    hidden = (256, 256)
    save_interval = 10_000

gamma, polyak = 0.99, 0.995
san_lr, q_lr = 5e-5, 5e-4
act_noise, target_noise, noise_clip = 0.1, 0.2, 0.5
policy_delay = 2
alpha = 1e-1
norm_clip_limit = 50

model_dir = f"./params_imit_warp_{gpu_id}"
video_dir = f"./videos_imit_warp_{gpu_id}"
logdir = f"runs/imit_warp_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{gpu_id}"
os.makedirs(model_dir, exist_ok=True)
os.makedirs(video_dir, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda", index=gpu_id)
else:
    raise RuntimeError("MuJoCo Warp 版は CUDA GPU が必要です")
torch.cuda.set_device(device)
print(f"使用デバイス: {device} ({torch.cuda.get_device_name(device)})")
print(f"環境数 {NUM_ENVS} / 物理プリセット {PHYSICS_PRESET} / 反復 {total_iters:,} "
      f"→ 遷移 {NUM_ENVS*total_iters:,} 本, 勾配更新 {updates_per_iter*total_iters:,} 回")


使用デバイス: cuda:0 (NVIDIA RTX A6000)
環境数 256 / 物理プリセット balanced / 反復 10,000 → 遷移 2,560,000 本, 勾配更新 80,000 回


## 5. 参照モーションのバッチ化

OmniTrot と同じ参照モーションを、全ワールド分まとめて GPU 上のテンソルとして計算するセル。

このセルに書かれているもの:

- `BatchTrot` — 全ワールド分の参照モーションを GPU 上のテンソルとしてまとめて計算するバッチ版生成器

In [5]:
_HX  = torch.tensor([l[1] for l in LEGS], dtype=torch.float64)
_HY  = torch.tensor([l[2] for l in LEGS], dtype=torch.float64)
_S   = torch.tensor([float(l[3]) for l in LEGS], dtype=torch.float64)
_OFF = torch.tensor([l[4] for l in LEGS], dtype=torch.float64)
# LEGS の並び (FL,FR,RL,RR) は QPOS_ORDER と同一なので、脚 i がそのまま qpos の i 番目に対応する


class BatchTrot:
    """指令 cmd[N,3] に対する参照モーションをまとめて生成する。

    cmd  : 胴体座標系の (vx, vy, ωz)
    t    : 時刻 [N] または [1] [s]
    h_swing: env ごとの遊脚高さ [N] (指令ゼロのワールドは 0 = 足を持ち上げない)

    入出力はすべて float64。脚の並びは QPOS_ORDER (FL,FR,RL,RR)。
    """

    def __init__(self, device, T=T_GAIT, beta=0.5, h0=H0):
        self.dev = device
        self.T, self.beta, self.h0 = T, beta, h0
        self.hx, self.hy, self.s, self.off = (t.to(device) for t in (_HX, _HY, _S, _OFF))

    # -- 胴体のワールド軌道 --
    def _base_pose(self, cmd, t):
        vx, vy, wz = cmd[..., 0], cmd[..., 1], cmd[..., 2]
        ps = wz * t
        small = wz.abs() < 1e-9
        wzs = torch.where(small, torch.ones_like(wz), wz)          # ゼロ割回避
        x = (vx * torch.sin(ps) + vy * (torch.cos(ps) - 1.0)) / wzs
        y = (vx * (1.0 - torch.cos(ps)) + vy * torch.sin(ps)) / wzs
        return torch.where(small, vx * t, x), torch.where(small, vy * t, y), ps

    # -- foothold: ストライド番号 n[...,4] → ワールド xy [...,4,2] --
    def _foothold(self, cmd, n):
        t_mid = (n - self.off + 0.5 * self.beta) * self.T
        x, y, ps = self._base_pose(cmd.unsqueeze(-2), t_mid)
        c, sn = torch.cos(ps), torch.sin(ps)
        nx, ny = self.hx, self.hy + self.s * D_HIP
        return torch.stack([x + c * nx - sn * ny, y + sn * nx + c * ny], dim=-1)

    # -- 足先のワールド位置 [...,4,3] --
    def _foot_world(self, cmd, t, h_swing):
        u = t.unsqueeze(-1) / self.T + self.off
        n = torch.floor(u)
        ph = u - n
        f0, f1 = self._foothold(cmd, n), self._foothold(cmd, n + 1.0)
        sw = ((ph - self.beta) / (1.0 - self.beta)).unsqueeze(-1)
        xy_sw = f0 + (f1 - f0) * 0.5 * (1.0 - torch.cos(np.pi * sw))
        z_sw = FOOT_R + h_swing.unsqueeze(-1) * torch.sin(np.pi * sw.squeeze(-1))
        stance = ph < self.beta
        xy = torch.where(stance.unsqueeze(-1), f0, xy_sw)          # 接地脚はワールド固定
        z = torch.where(stance, torch.full_like(z_sw, FOOT_R), z_sw)
        return torch.cat([xy, z.unsqueeze(-1)], dim=-1)

    # -- 3自由度 IK の解析解 (leg_ik のテンソル版) --
    @staticmethod
    def _leg_ik(p, s):
        px, py, pz = p[..., 0], p[..., 1], p[..., 2]
        d = s * D_HIP
        r = torch.hypot(py, pz)
        th1 = torch.atan2(pz, py) + torch.acos(torch.clamp(d / r, -1.0, 1.0))
        qz = -torch.sqrt(torch.clamp(r * r - d * d, min=1e-12))
        c3 = torch.clamp((px * px + qz * qz - L1 ** 2 - L2 ** 2) / (2 * L1 * L2), -1.0, 1.0)
        ph3 = -torch.acos(c3)
        th2 = torch.atan2(-px, -qz) - torch.atan2(L2 * torch.sin(ph3), L1 + L2 * torch.cos(ph3))
        return torch.stack([th1, th2, ph3 - DELTA], dim=-1)

    # -- 参照 qpos [N,19] --
    def qpos_at(self, cmd, t, h_swing):
        x, y, ps = self._base_pose(cmd, t)
        c, sn = torch.cos(ps), torch.sin(ps)
        hipw_x = x.unsqueeze(-1) + c.unsqueeze(-1) * self.hx - sn.unsqueeze(-1) * self.hy
        hipw_y = y.unsqueeze(-1) + sn.unsqueeze(-1) * self.hx + c.unsqueeze(-1) * self.hy
        fw = self._foot_world(cmd, t, h_swing)
        dx, dy, dz = fw[..., 0] - hipw_x, fw[..., 1] - hipw_y, fw[..., 2] - self.h0
        cu, su = c.unsqueeze(-1), sn.unsqueeze(-1)
        local = torch.stack([cu * dx + su * dy, -su * dx + cu * dy, dz], dim=-1)   # ワールド→胴体(ヨー)
        ang = self._leg_ik(local, self.s)                                          # [N,4,3]
        q = torch.zeros(cmd.shape[0], 19, dtype=cmd.dtype, device=cmd.device)
        q[:, 0], q[:, 1], q[:, 2] = x, y, self.h0
        q[:, 3], q[:, 6] = torch.cos(ps / 2), torch.sin(ps / 2)
        q[:, 7:] = ang.reshape(-1, 12)
        return q

    # -- 参照 qvel [N,18]: ベースは解析、関節は有限差分 --
    def qvel_at(self, cmd, t, h_swing, eps=1e-4):
        ps = cmd[:, 2] * t
        c, sn = torch.cos(ps), torch.sin(ps)
        v = torch.zeros(cmd.shape[0], 18, dtype=cmd.dtype, device=cmd.device)
        v[:, 0] = c * cmd[:, 0] - sn * cmd[:, 1]
        v[:, 1] = sn * cmd[:, 0] + c * cmd[:, 1]
        v[:, 5] = cmd[:, 2]
        q0 = self.qpos_at(cmd, t - eps, h_swing)
        q1 = self.qpos_at(cmd, t + eps, h_swing)
        v[:, 6:] = (q1[:, 7:] - q0[:, 7:]) / (2 * eps)
        return v

    # -- 1周期ぶんの関節参照テーブル: q_tab[N,CYC,12], dq_tab[N,CYC,12] --
    def joint_cycle(self, cmd, h_swing, fps=50, eps=1e-4):
        """1 ストライド周期の関節参照テーブル q[N,n,12], dq[N,n,12] (n = T*fps)。

        dq は中心差分 (t±eps)。3 本の時刻を 1 バッチにまとめて評価する。
        """
        n = int(round(self.T * fps))
        N = cmd.shape[0]
        t = (torch.arange(n, device=cmd.device, dtype=cmd.dtype) / fps).repeat(N)
        cmd_r = cmd.repeat_interleave(n, dim=0)
        hs_r = h_swing.repeat_interleave(n, dim=0)
        q_all = self.qpos_at(cmd_r.repeat(3, 1), torch.cat([t, t - eps, t + eps]),
                             hs_r.repeat(3))[:, 7:]
        M = N * n
        q = q_all[:M].reshape(N, n, 12)
        dq = ((q_all[2 * M:] - q_all[M:2 * M]) / (2 * eps)).reshape(N, n, 12)
        return q, dq


## 6. バッチ環境(MuJoCo Warp)

多数のワールドを GPU 上で並列に物理シミュレーションするバッチ環境のセル。低レベル PD 制御と状態書き込みは Warp カーネルとして GPU 上で実行する。

このセルに書かれているもの:

- `Go2ImitationWarpEnv` — 多数のワールドを並列シミュレーションするバッチ環境

In [6]:
# ---- warp カーネル ----
@wp.kernel
def _pd_kernel(qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
               target: wp.array2d(dtype=float), perm: wp.array(dtype=wp.int32),
               kp: float, kd: float, ctrl: wp.array2d(dtype=float)):
    """低レベル PD (物理刻みごと)。target はセンサ順、qpos/qvel は qpos 順なので perm で並べ替える。"""
    w, j = wp.tid()
    jj = perm[j]
    ctrl[w, j] = kp * (target[w, j] - qpos[w, 7 + jj]) - kd * qvel[w, 6 + jj]


@wp.kernel
def _write_state_kernel(qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
                        qacc_ws: wp.array2d(dtype=float),
                        new_q: wp.array2d(dtype=float), new_v: wp.array2d(dtype=float),
                        idx: wp.array(dtype=wp.int32), nq: int, nv: int):
    """idx で指定したワールドだけ状態を差し替える (auto-reset 用)。"""
    i = wp.tid()
    w = idx[i]
    for k in range(nq):
        qpos[w, k] = new_q[i, k]
    for k in range(nv):
        qvel[w, k] = new_v[i, k]
        qacc_ws[w, k] = 0.0                     # ソルバのウォームスタートも捨てる


# 物理ソルバの設定 (mjm.opt に上書きする値)
PHYSICS_PRESETS = {
    "exact": {},                                # go2.xml のまま: CG 100 反復 + elliptic + impratio 100
    # Newton 8 反復 + elliptic cone (既定)
    "balanced": dict(solver=mujoco.mjtSolver.mjSOL_NEWTON, iterations=8, ls_iterations=32),
    # Newton 1 反復 + pyramidal 近似 (最速だが接触の再現性は落ちる)
    "fast": dict(solver=mujoco.mjtSolver.mjSOL_NEWTON, iterations=1, ls_iterations=4,
                 cone=mujoco.mjtCone.mjCONE_PYRAMIDAL, impratio=1.0),
}


class Go2ImitationWarpEnv:
    """速度条件付き模倣学習環境の GPU バッチ版 (サーボ位置制御)。

    1 制御ステップ = frame_skip × timestep = 0.02 s (50 Hz)。その内側で PD を 500 Hz で回す。

      行動: [N,12] 立位姿勢からの目標関節角オフセット (ACT_SCALE 倍、可動域でクリップ)
      観測: [N,35] 関節 24 (角度/速度 × 12) + ジャイロ 3 + 加速度 3 + 正規化指令 3
            + 歩容位相 2 (sin, cos)
      報酬: 参照モーション追従 (DeepMimic 方式) — _compute_reward
      終了: 転倒 (_fallen) または max_ep_len 到達

    指令 (vx, vy, ωz) はワールドごとに持ち、リセット時に CMD_SCALE の範囲でサンプルする。
    戻り値はすべて device 上の torch テンソル。
    """

    # qpos順(FL,FR,RL,RR) → センサ/アクチュエータ順(FR,FL,RR,RL) の12関節置換
    PERM = np.array([3, 4, 5, 0, 1, 2, 9, 10, 11, 6, 7, 8])

    # info["reward_parts"] [N,7] の各列の意味 (そのまま TensorBoard のタグに使う)
    REWARD_KEYS = ("reward/pose", "reward/vel", "reward/cmd", "reward/yaw",
                   "penalty/dtarget", "penalty/height", "penalty/tilt")

    def __init__(self, num_envs, device, xml_path="scene_rough.xml", max_ep_len=500,
                 resample_cmd=True, preset=None, use_graph=None, seed=0, render_size=None):
        self.N = num_envs
        self.dev = device
        self.wp_dev = wp.get_device(f"cuda:{device.index}")
        self.max_steps = max_ep_len
        self.resample_cmd = resample_cmd
        self.frame_skip = 10
        self.rng = torch.Generator(device=device).manual_seed(seed)

        # ---- モデル: ctrl 制限を外して PD 出力トルクをそのまま印加する ----
        mjm = mujoco.MjModel.from_xml_path(xml_path)
        mjm.actuator_ctrllimited[:] = 0
        mjm.actuator_ctrlrange[:] = np.array([-1e6, 1e6])
        for k, v in PHYSICS_PRESETS[PHYSICS_PRESET if preset is None else preset].items():
            setattr(mjm.opt, k, v)
        # MuJoCo Warp は margin!=0 の box-box を MULTICCD/NATIVECCD 経路で扱えない
        # (go2.xml は margin=0.001、太ももが box)。margin は変えずに両フラグを切る
        mjm.opt.disableflags |= (mujoco.mjtDisableBit.mjDSBL_MULTICCD
                                 | mujoco.mjtDisableBit.mjDSBL_NATIVECCD)
        # 描画解像度に合わせてオフスクリーンバッファを広げる (既定は 640x480)
        self.render_w, self.render_h = render_size or VIDEO_PANEL
        mjm.vis.global_.offwidth = max(mjm.vis.global_.offwidth, self.render_w)
        mjm.vis.global_.offheight = max(mjm.vis.global_.offheight, self.render_h)
        self.mjm = mjm
        self.mjd = mujoco.MjData(mjm)                   # 描画用の CPU 側 data
        self.dt = self.frame_skip * mjm.opt.timestep    # 0.02 s (50 Hz)
        self.renderer = None                            # 遅延生成

        mujoco.mj_resetData(mjm, self.mjd)
        mujoco.mj_forward(mjm, self.mjd)
        with wp.ScopedDevice(self.wp_dev):
            self.m = mjw.put_model(mjm)
            self.m.opt.warn_overflow = 0 #エラー文非表示
            self.m.opt.graph_conditional = False        # 条件グラフノードは CUDA 12.4 未満で使えない
            self.d = mjw.put_data(mjm, self.mjd, nworld=self.N)
            self.perm_wp = wp.array(self.PERM.astype(np.int32), dtype=wp.int32)
            self.target_wp = wp.zeros((self.N, 12), dtype=float)
        # warp 配列の torch view (ゼロコピー: 書き込みはそのまま物理側に反映される)
        self.qpos = wp.to_torch(self.d.qpos)            # [N,19]
        self.qvel = wp.to_torch(self.d.qvel)            # [N,18]
        self.sensordata = wp.to_torch(self.d.sensordata)
        self.target = wp.to_torch(self.target_wp)       # [N,12] センサ順の目標角

        # ---- センサのアドレス ----
        self.joints = ['FR_hip', 'FR_thigh', 'FR_calf', 'FL_hip', 'FL_thigh', 'FL_calf',
                       'RR_hip', 'RR_thigh', 'RR_calf', 'RL_hip', 'RL_thigh', 'RL_calf']
        adr = lambda n: mjm.sensor_adr[mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_SENSOR, n)]
        self.a_pos = adr(f"{self.joints[0]}_pos")       # 以降 12 個が joints の順に並んでいる
        self.a_vel = adr(f"{self.joints[0]}_vel")
        self.a_gyro, self.a_acc = adr("imu_gyro"), adr("imu_acc")

        # ---- 定数をテンソル化 ----
        t32 = lambda x: torch.as_tensor(x, dtype=torch.float32, device=device)
        self.perm = torch.as_tensor(self.PERM, dtype=torch.long, device=device)
        self.cmd_scale = t32(CMD_SCALE)
        self.act_scale = t32(ACT_SCALE)
        self.standing = t32(standing_pose(H0))                # qpos順の立位関節角
        self.default_sens = self.standing[self.perm]             # センサ順に変換
        self.jnt_lo = t32(mjm.jnt_range[1:, 0])[self.perm]       # 目標角の可動域 (センサ順)
        self.jnt_hi = t32(mjm.jnt_range[1:, 1])[self.perm]

        # ---- env ごとの状態 ----
        self.trot = BatchTrot(device)
        self.cmd = torch.zeros(self.N, 3, device=device)
        self.h_swing = torch.zeros(self.N, device=device)
        self.q_tab = torch.zeros(self.N, CYC, 12, device=device)
        self.dq_tab = torch.zeros(self.N, CYC, 12, device=device)
        self.k0 = torch.zeros(self.N, dtype=torch.long, device=device)
        self.step_count = torch.zeros(self.N, dtype=torch.long, device=device)
        self.prev_target = torch.zeros(self.N, 12, device=device)
        self.last_target = torch.zeros(self.N, 12, device=device)
        self.all_idx = torch.arange(self.N, device=device)

        self.obs_dim, self.act_dim, self.act_limit = 35, 12, 1.0
        self.graph = self.fwd_graph = None
        use_graph = USE_CUDA_GRAPH if use_graph is None else use_graph
        if use_graph:
            self._build_graph()

    # ---------------- warp 実行 (torch と同じ CUDA ストリームに載せて順序を保証する) ----------------
    def _stream(self):
        return wp.stream_from_torch(torch.cuda.current_stream(self.dev))

    def _launch_pd(self):
        wp.launch(_pd_kernel, dim=(self.N, 12),
                  inputs=[self.d.qpos, self.d.qvel, self.target_wp, self.perm_wp,
                          float(SERVO_KP), float(SERVO_KD)],
                  outputs=[self.d.ctrl], device=self.wp_dev)

    def _build_graph(self):
        """(PD → mj_step) × frame_skip と mjw.forward を CUDA グラフに固める。"""
        with wp.ScopedDevice(self.wp_dev):
            for _ in range(2):                          # カーネルを事前コンパイル/ロード
                self._launch_pd()
                mjw.step(self.m, self.d)
            mjw.forward(self.m, self.d)
            wp.synchronize()
            with wp.ScopedCapture() as cap:
                for _ in range(self.frame_skip):
                    self._launch_pd()
                    mjw.step(self.m, self.d)
            self.graph = cap.graph
            with wp.ScopedCapture() as cap_fwd:
                mjw.forward(self.m, self.d)
            self.fwd_graph = cap_fwd.graph

    def _physics(self):
        if self.graph is not None:
            wp.capture_launch(self.graph, stream=self._stream())
        else:
            with wp.ScopedDevice(self.wp_dev), wp.ScopedStream(self._stream()):
                for _ in range(self.frame_skip):
                    self._launch_pd()
                    mjw.step(self.m, self.d)

    def _forward(self):
        """qpos/qvel を書き換えた後に派生量 (センサ等) を作り直す。"""
        if self.fwd_graph is not None:
            wp.capture_launch(self.fwd_graph, stream=self._stream())
        else:
            with wp.ScopedDevice(self.wp_dev), wp.ScopedStream(self._stream()):
                mjw.forward(self.m, self.d)

    # ---------------- 指令 ----------------
    def _sample_cmd(self, n):
        c = (torch.rand(n, 3, device=self.dev, generator=self.rng) * 2 - 1) * self.cmd_scale
        still = torch.rand(n, 1, device=self.dev, generator=self.rng) < 0.15   # 15% は静止指令
        return torch.where(still, torch.zeros_like(c), c)

    def _set_cmd(self, idx, cmd):
        """idx のワールドに指令を設定し、参照テーブル q_tab/dq_tab を作り直す。"""
        cmd = torch.as_tensor(cmd, dtype=torch.float32, device=self.dev).reshape(-1, 3).clone()
        if cmd.shape[0] == 1:
            cmd = cmd.expand(idx.numel(), 3).clone()
        dead = (cmd / self.cmd_scale).norm(dim=1) < 0.05                       # デッドバンド
        cmd[dead] = 0.0
        moving = cmd.abs().any(dim=1)
        hs = torch.where(moving, torch.full_like(cmd[:, 0], H_SWING), torch.zeros_like(cmd[:, 0]))
        self.cmd[idx], self.h_swing[idx] = cmd, hs
        q, dq = self.trot.joint_cycle(cmd.double(), hs.double(), fps=int(round(1 / self.dt)))
        self.q_tab[idx], self.dq_tab[idx] = q.float(), dq.float()

    # ---------------- エピソード ----------------
    def _reset_idx(self, idx, command=None):
        n = idx.numel()
        if n == 0:
            return
        self._set_cmd(idx, self._sample_cmd(n) if command is None else command)
        self.step_count[idx] = 0
        self.k0[idx] = torch.randint(CYC, (n,), device=self.dev, generator=self.rng)  # RSI
        t0 = (self.k0[idx] * self.dt).double()
        cmd_d, hs_d = self.cmd[idx].double(), self.h_swing[idx].double()
        qp = self.trot.qpos_at(cmd_d, t0, hs_d).float()
        qv = self.trot.qvel_at(cmd_d, t0, hs_d).float()
        qp[:, 7:] += torch.rand(n, 12, device=self.dev, generator=self.rng) * 0.1 - 0.05  # 参照まわりの摂動
        with wp.ScopedDevice(self.wp_dev), wp.ScopedStream(self._stream()):
            wp.launch(_write_state_kernel, dim=n,
                      inputs=[self.d.qpos, self.d.qvel, self.d.qacc_warmstart,
                              wp.from_torch(qp.contiguous()), wp.from_torch(qv.contiguous()),
                              wp.from_torch(idx.to(torch.int32).contiguous(), dtype=wp.int32), 19, 18],
                      device=self.wp_dev)
        self.prev_target[idx] = qp[:, 7:][:, self.perm]        # Δ目標角ペナルティの基準
        self.last_target[idx] = self.prev_target[idx]

    def reset(self, command=None):
        """全ワールドをリセット。command は [3] (全ワールド共通) か [N,3] (ワールドごと)。"""
        self._reset_idx(self.all_idx, command)
        self._forward()
        return self._get_obs()

    def autoreset(self, done, command=None):
        """done のワールドだけリセットし、最新の観測を返す。"""
        idx = self.all_idx[done]
        if idx.numel():
            self._reset_idx(idx, command)
            self._forward()
        return self._get_obs()

    def _phase_k(self):
        return (self.k0 + self.step_count) % CYC

    def step(self, action):
        """リセット前の (obs2, reward, done, info) を返す。done の処理は autoreset() 側。"""
        target = self.default_sens + action.to(self.dev, torch.float32) * self.act_scale
        self.last_target = torch.clamp(target, self.jnt_lo, self.jnt_hi)
        self.target.copy_(self.last_target)                # warp 側の目標角バッファへ (グラフが参照)
        self._physics()                                    # PD 500Hz × frame_skip
        self.step_count += 1
        if self.resample_cmd:      # エピソード中盤で確率0.5で指令を切り替え → 速度遷移も学習
            sel = (self.step_count == self.max_steps // 2) & \
                  (torch.rand(self.N, device=self.dev, generator=self.rng) < 0.5)
            idx = self.all_idx[sel]
            if idx.numel():
                self._set_cmd(idx, self._sample_cmd(idx.numel()))
        fallen = self._fallen()
        reward, parts = self._compute_reward(fallen)
        timeout = self.step_count >= self.max_steps
        return self._get_obs(), reward, fallen | timeout, {
            "reward_parts": parts, "fallen": fallen, "timeout": timeout, "cmd": self.cmd}

    # ---------------- 観測 (35次元) ----------------
    def _get_obs(self):
        sd = self.sensordata
        ang = sd[:, self.a_pos:self.a_pos + 12] - self.default_sens
        vel = sd[:, self.a_vel:self.a_vel + 12]
        ph = 2 * np.pi * self._phase_k().float() / CYC
        return torch.cat([
            torch.stack([ang, vel], dim=-1).reshape(self.N, 24),        # 24 (関節ごとに角度/速度)
            sd[:, self.a_gyro:self.a_gyro + 3],                         # 3
            sd[:, self.a_acc:self.a_acc + 3],                           # 3
            self.cmd / self.cmd_scale,                                  # 3 (正規化指令)
            torch.stack([torch.sin(ph), torch.cos(ph)], dim=-1),        # 2 (歩容位相)
        ], dim=1)

    # ---------------- 報酬 ----------------
    @staticmethod
    def _quat2mat(q):
        w, x, y, z = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
        return torch.stack([
            1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w),
            2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w),
            2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y),
        ], dim=1).reshape(-1, 3, 3)

    def _compute_reward(self, fallen):
        """r = 0.5·姿勢 + 0.10·関節速度 + 0.20·速度指令 + 0.20·ヨー指令 − ペナルティ (転倒時は −1)。

        内訳は parts[N,7] (REWARD_KEYS の順) として info["reward_parts"] で返す。
        """
        k = self._phase_k()
        q, dq = self.qpos[:, 7:], self.qvel[:, 6:]
        e_pose = ((q - self.q_tab[self.all_idx, k]) ** 2).sum(1)
        e_vel = ((dq - self.dq_tab[self.all_idx, k]) ** 2).sum(1)

        Rm = self._quat2mat(self.qpos[:, 3:7])
        v_body = torch.einsum('nij,nj->ni', Rm.transpose(1, 2), self.qvel[:, 0:3])
        wz = self.qvel[:, 5]

        r_pose = 0.6 * torch.exp(-5.0 * e_pose)
        r_vel = 0.10 * torch.exp(-0.05 * e_vel)
        r_cmd = 0.15 * torch.exp(-((v_body[:, :2] - self.cmd[:, :2]) ** 2).sum(1) / 0.01)
        r_yaw = 0.15 * torch.exp(-(wz - self.cmd[:, 2]) ** 2 / 0.01)

        dtar = self.last_target - self.prev_target
        self.prev_target = self.last_target.clone()
        tilt2 = self.qpos[:, 4] ** 2 + self.qpos[:, 5] ** 2
        p_dtar = 0.3 * ((dtar / self.act_scale) ** 2).sum(1)            # Δ目標角
        p_height = 5.0 * (self.qpos[:, 2] - H0) ** 2                    # 胴体高さ
        p_tilt = 5.0 * tilt2                                            # 傾き

        parts = torch.stack([r_pose, r_vel, r_cmd, r_yaw,
                             p_dtar, p_height, p_tilt], dim=1)          # REWARD_KEYS の順
        rew = r_pose + r_vel + r_cmd + r_yaw - (p_dtar + p_height + p_tilt)
        return torch.where(fallen, torch.full_like(rew, -1.0), rew), parts

    def _fallen(self):
        """胴体高さ < 0.23 m または傾き > 0.25 で転倒とみなす。"""
        return (self.qpos[:, 2] < 0.23) | (torch.hypot(self.qpos[:, 4], self.qpos[:, 5]) > 0.25)

    def close(self):
        """描画用の EGL コンテキストを解放する (プロセス終了前に必ず呼ぶこと)。"""
        if self.renderer is not None:
            self.renderer.close()
            self.renderer = None

    # ---------------- 描画 (指定ワールドを CPU 側 data に取り出してレンダリング) ----------------
    def render(self, world=0, camera="track"):
        """world のワールドを render_size で描画して [H,W,3] の uint8 を返す。"""
        if self.renderer is None:
            torch.cuda.empty_cache()                    # フレームバッファぶんの VRAM を空ける
            self.renderer = mujoco.Renderer(self.mjm, self.render_h, self.render_w)
        mjw.get_data_into(self.mjd, self.mjm, self.d, world_id=world)
        mujoco.mj_forward(self.mjm, self.mjd)
        self.renderer.update_scene(self.mjd, camera=camera)
        return self.renderer.render()


## 7. リプレイバッファとネットワーク定義(GPU バッチ版)

学習に使うデータ構造とニューラルネットワークを定義するセル。すべて num_envs ワールドぶんをまとめて処理する GPU 常駐のバッチ版。

このセルに書かれているもの:

- `ReplayBuffer` — GPU 常駐のリング型経験再生バッファ(観測の逐次正規化つき)
- `PopSpike` — ポピュレーション符号化の発火関数(代理勾配つき)
- `SpikeFn` — 発火のステップ関数 + 代理勾配
- `lif_step()` — LIF ニューロンの1ステップ更新
- `MLP` — 3層 ReLU の全結合ネット
- `MLPQFunction` — 観測と行動から価値を出す Q 関数(批評家)
- `Encoder` — 観測 → スパイク列の変換(ポピュレーション符号化)
- `Decoder` — スパイク発火率 → 行動 (-1〜1) の復号
- `SpikeActor` — 3層 LIF のスパイキング方策(膜電位を制御ステップ間で持ち越す)
- `Policy` — アクターと双子 Q をまとめる入れ物
- `load_san_state_dict()` — チェックポイントをアクターへ読み込むヘルパ

In [7]:
class ReplayBuffer:
    """GPU 常駐のリング型リプレイバッファ。1 反復で num_envs 本の遷移をまとめて格納する。"""

    def __init__(self, obs_dim, act_dim, mem_dim, size, clip_limit, device):
        z = lambda *s: torch.zeros(*s, dtype=torch.float32, device=device)
        self.obs_buf, self.obs2_buf = z(size, obs_dim), z(size, obs_dim)
        self.act_buf = z(size, act_dim)
        self.rew_buf, self.done_buf = z(size), z(size)
        self.mem_buf, self.mem2_buf = z(size, mem_dim), z(size, mem_dim)
        self.ptr, self.size, self.max_size = 0, 0, size
        self.clip_limit = clip_limit
        self.dev = device
        self.eps = torch.finfo(torch.float32).eps
        self.mean, self.var = z(obs_dim), torch.ones(obs_dim, device=device)
        self.total_count = self.eps
        gb = (2 * obs_dim + act_dim + 2 + 2 * mem_dim) * 4 * size / 2 ** 30
        print(f"リプレイバッファ: {size:,} 遷移 × {(2*obs_dim+act_dim+2+2*mem_dim)*4/1024:.1f} KB "
              f"= {gb:.2f} GB (GPU)")

    def store_batch(self, obs, act, rew, next_obs, done, mem, mem2):
        n = obs.shape[0]
        idx = (self.ptr + torch.arange(n, device=self.dev)) % self.max_size
        self.obs_buf[idx], self.obs2_buf[idx] = obs, next_obs
        self.act_buf[idx] = act
        self.rew_buf[idx], self.done_buf[idx] = rew, done
        self.mem_buf[idx], self.mem2_buf[idx] = mem, mem2
        self.ptr = (self.ptr + n) % self.max_size
        self.size = min(self.size + n, self.max_size)
        self._update_norm(obs)

    def _update_norm(self, obs):
        """観測の平均・分散を並列分散マージで更新する (1 バッチ = 1 反復ぶんの N 本)。"""
        n = obs.shape[0]
        batch_mean, batch_var = obs.mean(0), obs.var(0, unbiased=False)
        tmp_total_count = self.total_count + n
        delta_mean = batch_mean - self.mean
        self.mean = self.mean + delta_mean * (n / tmp_total_count)
        m_a = self.var * self.total_count
        m_b = batch_var * n
        m_2 = m_a + m_b + delta_mean ** 2 * self.total_count * n / tmp_total_count
        self.var = m_2 / tmp_total_count
        self.total_count = tmp_total_count

    def sample_batch(self, batch_size):
        idxs = torch.randint(0, self.size, (batch_size,), device=self.dev)
        return dict(
            obs=self.normalize_obs(self.obs_buf[idxs]),
            obs2=self.normalize_obs(self.obs2_buf[idxs]),
            act=self.act_buf[idxs],
            rew=self.rew_buf[idxs],
            done=self.done_buf[idxs],
            mem=self.mem_buf[idxs],
            mem2=self.mem2_buf[idxs],
        )

    def normalize_obs(self, obs):
        return torch.clamp((obs - self.mean) / torch.sqrt(self.var + self.eps),
                           -self.clip_limit, self.clip_limit)


# ==================== スパイク生成と LIF ====================
# beta: 膜電位の減衰率 / threshold: 発火閾値 / slope: 代理勾配の鋭さ / ENC_VTH: 符号化の閾値
LIF_BETA, LIF_THRESHOLD, SPIKE_SLOPE = 0.5, 1.0, 3.0
ENC_VTH = 0.999


class PopSpike(torch.autograd.Function):
    """population 符号化 + 確率的スパイク生成。

    forward : spk[b, j*P+k] = 1[p[b,j] + u[b,j,k] > vth]
    backward: dL/dp[b,j] = Σ_k g[b,j,k] / (slope*|v-vth| + 1)^2

    u (一様乱数) を引数で受け取るので、外から乱数列を固定できる。
    """

    @staticmethod
    def forward(ctx, p, u, slope, vth):
        v = p.unsqueeze(-1) + u
        ctx.save_for_backward(v)
        ctx.slope, ctx.vth = slope, vth
        return v.gt(vth).to(p.dtype).reshape(p.shape[0], -1)

    @staticmethod
    def backward(ctx, grad_output):
        (v,) = ctx.saved_tensors
        denom = (ctx.slope * (v - ctx.vth).abs() + 1.0) ** 2
        return (grad_output.view(v.shape) / denom).sum(-1), None, None, None


class SpikeFn(torch.autograd.Function):
    """ヘヴィサイド発火 + fast-sigmoid 代理勾配 1/(slope*|x|+1)^2。"""

    @staticmethod
    def forward(ctx, x, slope):
        ctx.save_for_backward(x)
        ctx.slope = slope
        return (x > 0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output / (ctx.slope * x.abs() + 1.0) ** 2, None


def lif_step(cur, mem_prev, beta=LIF_BETA, threshold=LIF_THRESHOLD, slope=SPIKE_SLOPE):
    """LIF (reset = subtract) の 1 ステップ。

      reset = 1[mem_prev > thr]                  (勾配は通さない)
      mem   = beta*mem_prev + cur - reset*thr
      spk   = FastSigmoid(mem - thr)
    """
    reset = (mem_prev > threshold).to(mem_prev.dtype)
    mem = beta * mem_prev + cur - reset * threshold
    return SpikeFn.apply(mem - threshold, slope), mem


class MLP(nn.Module):
    """3 層 MLP (ReLU、出力は線形)。sizes = [in, h1, h2, out]。"""

    def __init__(self, sizes):
        super().__init__()
        self.Linear1 = nn.Linear(sizes[0], sizes[1])
        self.Linear2 = nn.Linear(sizes[1], sizes[2])
        self.Linear3 = nn.Linear(sizes[2], sizes[3])
        self.activation = nn.ReLU()
        self.output_activation = nn.Identity()
        nn.init.kaiming_normal_(self.Linear1.weight)
        nn.init.kaiming_normal_(self.Linear2.weight)
        nn.init.kaiming_normal_(self.Linear3.weight)

    def forward(self, x):
        x = self.activation(self.Linear1(x))
        x = self.activation(self.Linear2(x))
        return self.output_activation(self.Linear3(x))


class MLPQFunction(nn.Module):
    """Q(s, a) → [B]。"""

    def __init__(self, obs_dim, act_dim, hidden_sizes):
        super().__init__()
        self.q = MLP([obs_dim + act_dim] + [hidden_sizes[0]] + [hidden_sizes[1]] + [1])

    def forward(self, obs, act):
        q = self.q(torch.cat([obs, act], dim=-1))
        return torch.squeeze(q, -1)


class Encoder(nn.Module):
    """観測を population 符号化して確率的スパイク列にする ([B,obs_dim] → [B,obs_dim*2*pop_dim])。"""

    def __init__(self, obs_dim, pop_dim, device):
        super().__init__()
        self.obs_dim = obs_dim
        self.pop_dim = pop_dim
        self.device = device
        self.activation = nn.Tanh()
        self.zeros = torch.zeros([1, self.obs_dim*2], device=self.device)
        self.weight = nn.Parameter(torch.ones(1, self.obs_dim, device=self.device))
        self.bias = nn.Parameter(torch.zeros(1, self.obs_dim, device=self.device))
        self.fixed_noise = None            # テンソルを入れるとスパイク生成の乱数の代わりに使う

    def forward(self, obs):
        obs = self.activation(obs*self.weight + self.bias)   # 学習可能なゲイン・バイアスで ±1 に圧縮
        # ±チャネルへ分解して pop_dim 個に複製し、乱数としきい値比較 → 値に比例した発火確率
        p = torch.maximum(torch.cat([obs, -obs], dim=1), self.zeros)   # 正/負チャネルの半波整流
        u = (self.fixed_noise if self.fixed_noise is not None else
             torch.rand(p.shape[0], self.obs_dim*2, self.pop_dim, device=p.device, dtype=p.dtype))
        return PopSpike.apply(p, u, SPIKE_SLOPE, ENC_VTH)


class Decoder(nn.Module):
    """スパイク列 → 行動 [B,act_dim]。population の発火率の差を tanh に通す。"""

    def __init__(self, act_dim, pop_dim, device):
        super().__init__()
        self.act_dim = act_dim
        self.pop_dim = pop_dim
        self.activation = nn.Tanh()

        self.weight = nn.Parameter(torch.ones(1, act_dim, device=device), requires_grad=False)
        self.bias = nn.Parameter(torch.zeros(1, act_dim, device=device), requires_grad=False)

    def forward(self, spk):
        s = spk.reshape([-1, self.act_dim*2, self.pop_dim]).sum(-1)/self.pop_dim   # ポピュレーションごとの発火率
        return self.activation((s[..., :self.act_dim]-s[..., self.act_dim:])*self.weight + self.bias)


class SpikeActor(nn.Module):
    """スパイキング actor: Encoder → (Linear + LIF) ×3 → Decoder。

    膜電位 m[N,mem_dim] は 3 層ぶんを連結したもので、呼び出し側が状態として持ち回す。
    """

    def __init__(self, obs_dim, hidden_sizes, act_dim):
        super().__init__()
        self.Linear1 = nn.Linear(obs_dim*2*encoder_pop_dim, hidden_sizes[0])
        self.Linear2 = nn.Linear(hidden_sizes[0], hidden_sizes[1])
        self.Linear3 = nn.Linear(hidden_sizes[1], hidden_sizes[2])
        self.encoder = Encoder(obs_dim, encoder_pop_dim, device)
        self.decoder = Decoder(act_dim, decoder_pop_dim, device)
        self.p1 = hidden_sizes[0]
        self.p2 = hidden_sizes[0] + hidden_sizes[1]

        nn.init.kaiming_normal_(self.Linear1.weight)
        nn.init.kaiming_normal_(self.Linear2.weight)
        nn.init.kaiming_normal_(self.Linear3.weight)

        self.record_spikes = False         # True にすると出力層のスパイクを spk3_hist に貯める
        self.spk3_hist = []

    def forward(self, x, m):
        spk1, mem1 = lif_step(self.Linear1(self.encoder(x)), m[:, :self.p1])
        spk2, mem2 = lif_step(self.Linear2(spk1), m[:, self.p1:self.p2])
        spk3, mem3 = lif_step(self.Linear3(spk2), m[:, self.p2:])

        action = self.decoder(spk3)
        mem_out = torch.cat([mem1, mem2, mem3], dim=1)

        if self.record_spikes:
            self.spk3_hist.append(spk3.detach().cpu().numpy())

        return action, mem_out

    def reset_spike_records(self):
        self.spk3_hist = []


class Policy(nn.Module):
    """スパイキング actor と TD3 用の Q 関数 2 本をまとめたもの。"""

    def __init__(self):
        super().__init__()
        self.san = SpikeActor(obs_dim, hidden_sizes, act_dim)
        self.q1 = MLPQFunction(obs_dim, act_dim, hidden_sizes)
        self.q2 = MLPQFunction(obs_dim, act_dim, hidden_sizes)

    def forward(self, x, m):
        a, m2 = self.san(x, m)
        return a, m2

    def act(self, obs, m):
        """行動と次の膜電位を GPU 上のテンソルのまま返す。"""
        with torch.no_grad():
            a, m2 = self.san(obs, m)
            return a, m2


def load_san_state_dict(san, path_or_sd):
    """actor の重みを読み込む (パス、または state_dict)。

    state_dict に LIF の定数バッファ (lif*.beta / threshold など) が含まれている場合は、
    値がこの実装の定数と一致することを確かめたうえで取り除く。
    """
    sd = (torch.load(path_or_sd, weights_only=True) if isinstance(path_or_sd, str)
          else dict(path_or_sd))
    expect = {"beta": LIF_BETA, "threshold": LIF_THRESHOLD,
              "graded_spikes_factor": 1.0, "reset_mechanism_val": 0.0}
    for k in [k for k in sd if k.startswith("lif")]:
        name = k.split(".", 1)[1]
        v = float(sd.pop(k))
        if name in expect and abs(v - expect[name]) > 1e-6:
            raise ValueError(f"{k}={v} が本実装の想定 ({expect[name]}) と違います")
    san.load_state_dict(sd)
    return san


## 8. TD3 の損失関数と評価ユーティリティ

学習の「更新則」と「評価」に使う関数をまとめたセル。

このセルに書かれているもの:

- `compute_loss_q()` — 双子 Q の TD 誤差(TD3)
- `compute_loss_san()` — アクター損失(Q の最大化 + 内部状態の正則化)
- `update()` — 1回分の TD3 更新(Q は毎回、方策とターゲット網は policy_delay 回に1回)
- `get_action()` — 方策出力に探索ノイズを加えた行動選択
- `new_mem()` — SNN 内部状態(膜電位)の初期化
- `grid_frame()` — 複数ワールドの描画をタイル状に並べるユーティリティ
- `test_agent()` — 固定指令セット `TEST_CMDS` での評価
- `save_training_video()` — 学習途中の方策の動画保存

In [8]:
# ---------------- TD3 の損失と更新 ----------------

def compute_loss_q(data):
    """target policy smoothing + clipped double-Q の TD 誤差 (Q1, Q2 の和)。"""
    o, a, r, o2, d = data['obs'], data['act'], data['rew'], data['obs2'], data['done']
    m2 = data['mem2']
    q1, q2 = ac.q1(o, a), ac.q2(o, a)
    with torch.no_grad():
        san_targ, _ = ac_targ.san(o2, m2)
        eps = torch.clamp(torch.randn_like(san_targ) * target_noise, -noise_clip, noise_clip)
        a2 = torch.clamp(san_targ + eps, -act_limit, act_limit)      # ノイズ付きターゲット行動
        q_targ = torch.min(ac_targ.q1(o2, a2), ac_targ.q2(o2, a2))   # 2本の小さい方 → 過大評価を抑制
        backup = r + gamma * (1 - d) * q_targ                        # TD ターゲット
    return ((q1 - backup) ** 2).mean() + ((q2 - backup) ** 2).mean()


def compute_loss_san(data):
    """-Q1 + alpha·(膜電位)^2 : 膜電位の発散を抑える正則化つき。"""
    o, m = data['obs'], data['mem']
    a, m2 = ac.san(o, m)
    return -ac.q1(o, a).mean() + alpha * (m2 ** 2).mean()


def update(data, timer):
    """1 回の勾配更新。(critic 損失, actor 損失) を返す (actor は更新しなかった反復では None)。"""
    loss_san = None
    q_optimizer.zero_grad()
    loss_q = compute_loss_q(data)
    loss_q.backward()
    q_optimizer.step()
    if timer % policy_delay == 0:                  # actor と target は policy_delay 回に 1 回
        for p in q_params:
            p.requires_grad = False
        san_optimizer.zero_grad()
        loss_san = compute_loss_san(data)
        loss_san.backward()
        san_optimizer.step()
        for p in q_params:
            p.requires_grad = True
        with torch.no_grad():                      # polyak 更新
            torch._foreach_mul_(ac_targ_params, polyak)
            torch._foreach_add_(ac_targ_params, ac_params, alpha=1 - polyak)
    return loss_q.detach(), (loss_san.detach() if loss_san is not None else None)


def get_action(o, m, noise_scale):
    """o は正規化済みの観測テンソル [N,obs_dim]。行動 [N,act_dim] と次の膜電位を返す。"""
    a, m2 = ac.act(o, m)
    if noise_scale:
        a = a + noise_scale * torch.randn_like(a)
    return torch.clamp(a, -act_limit, act_limit), m2


def new_mem(n):
    return torch.normal(0, 1.0, size=[n, mem_dim], dtype=torch.float32, device=device)


# ---------------- 固定指令での評価と動画 ----------------
# 4 指令を 4 ワールドで同時に走らせる (TEST_CMD_MAT の行番号 = ワールド番号)
TEST_CMDS = {
    "前進":  np.array([1.00, 0.0, 0.0]),
    "後退":  np.array([-1.00, 0.0, 0.0]),
    "左移動": np.array([0.0, 1.00, 0.0]),
    "左旋回": np.array([0.0, 0.0, 1.00]),
}
TEST_CMD_MAT = np.stack(list(TEST_CMDS.values()))

VIDEO_TASKS = list(TEST_CMDS)                      # 動画の 2x2 配置順 (左上→右上→左下→右下)
VIDEO_LABELS = {"前進": "forward", "後退": "backward", "左移動": "left", "左旋回": "turn left"}
VIDEO_WORLDS = [list(TEST_CMDS).index(n) for n in VIDEO_TASKS]
_VIDEO_FONT = ImageFont.load_default(size=22)      # 既定フォントは日本語が出ないのでラベルはローマ字


def grid_frame(env, fallen=None):
    """4 タスクのワールドを描画し、タスク名と指令を焼き込んで 2x2 に並べた 1 フレームを返す。

    fallen[N] (bool) を渡すと、転倒したワールドにだけ FALLEN を表示する。
    時間切れ (max_ep_len 到達) は転倒ではないので表示しない。
    """
    panels = []
    for name, w in zip(VIDEO_TASKS, VIDEO_WORLDS):
        img = Image.fromarray(env.render(world=w))
        cmd = TEST_CMD_MAT[w]
        down = False if fallen is None else bool(fallen[w])
        txt = (f"{VIDEO_LABELS[name]}  [{cmd[0]:+.2f} {cmd[1]:+.2f} {cmd[2]:+.2f}]"
               + ("  FALLEN" if down else ""))
        ImageDraw.Draw(img).text((10, 8), txt, font=_VIDEO_FONT, fill=(255, 255, 255),
                                 stroke_width=2, stroke_fill=(0, 0, 0))
        panels.append(np.asarray(img))
    return np.concatenate([np.concatenate(panels[:2], axis=1),
                           np.concatenate(panels[2:], axis=1)], axis=0)


def test_agent(video_path=None):
    """評価環境 (1 ワールド = 1 指令) を 1 エピソード回し、指令ごとの報酬合計を返す。

    video_path を渡すと 4 タスクを 2x2 に並べた動画をそこへ逐次書き出す。
    """
    o = test_env.reset(command=TEST_CMD_MAT)
    m = new_mem(test_env.N)
    ep_ret = torch.zeros(test_env.N, device=device)
    alive = torch.ones(test_env.N, dtype=torch.bool, device=device)
    fell = torch.zeros(test_env.N, dtype=torch.bool, device=device)   # 一度転倒したら立てたまま
    writer_v = imageio.get_writer(video_path, fps=50) if video_path else None
    n_frames = 0
    try:
        for _ in range(test_env.max_steps):
            a, m2 = get_action(replay_buffer.normalize_obs(o), m, 0)
            o, r, d, info = test_env.step(a)
            ep_ret += r * alive                  # 終了済みワールドは加算しない
            fell |= info["fallen"]
            alive &= ~d
            m = m2
            if writer_v is not None:
                writer_v.append_data(grid_frame(test_env, fell))    # 転倒後もそのまま映す
                n_frames += 1
            if not bool(alive.any()):
                break
    finally:
        if writer_v is not None:
            writer_v.close()
    rets = {name: round(v, 1) for name, v in zip(TEST_CMDS, ep_ret.tolist())}
    return (rets, n_frames) if video_path else rets


def save_training_video(it):
    """学習途中の方策で 4 タスクを 1 本の動画にまとめて保存する。

    ffmpeg は一時ファイルへ書き出し、完成してから video_dir へ移す。
    書き出しに失敗した場合はメッセージを出して学習を続行する。
    """
    path = f"{video_dir}/imit_warp_{it//1000}Kit_4tasks.mp4"
    tmp = os.path.join(tempfile.gettempdir(), os.path.basename(path))
    try:
        rets, n_frames = test_agent(video_path=tmp)
        shutil.move(tmp, path)
    except Exception as e:
        print(f"  動画保存に失敗 (学習は継続): {type(e).__name__}: {e}")
        traceback.print_exc(limit=3)
        return
    scores = " / ".join(f"{n}={rets[n]}" for n in VIDEO_TASKS)
    print(f"  動画保存: {path} ({n_frames}フレーム, 報酬 {scores})")


## 9. 環境の生成

学習用と評価用のバッチ環境を作るセル。学習用は指令をエピソード中盤で再サンプルし、評価用は固定指令セットを1ワールドに1指令ずつ割り当てる。

In [9]:
# 学習用は指令をエピソード中盤で再サンプルする。評価用は TEST_CMDS を 1 ワールド 1 指令で固定
env = Go2ImitationWarpEnv(NUM_ENVS, device, max_ep_len=max_ep_len, resample_cmd=True, seed=0)
test_env = Go2ImitationWarpEnv(len(TEST_CMDS), device, max_ep_len=max_ep_len, resample_cmd=False, seed=1)

seed = 0
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

obs_dim = env.obs_dim
act_dim = env.act_dim
act_limit = env.act_limit

hidden_sizes = (hidden[0], hidden[1], act_dim*2*decoder_pop_dim)
mem_dim = sum(hidden_sizes)                  # LIF 3 層ぶんの膜電位を連結した次元
print(f"obs_dim={obs_dim} act_dim={act_dim} mem_dim={mem_dim}")

test_env.render(world=0)                     # 描画を先に初期化しておく
# EGL コンテキストの解放は mujoco 内部の eglTerminate より先に走らせる (atexit は LIFO)
atexit.register(env.close)
atexit.register(test_env.close)


obs_dim=35 act_dim=12 mem_dim=6656


<bound method Go2ImitationWarpEnv.close of <__main__.Go2ImitationWarpEnv object at 0x7f20709d4b50>>

## 10. 学習ループ

ネットワーク・オプティマイザ・リプレイバッファを作り、TD3 で学習するメインのセル。毎反復、全ワールドの遷移をまとめてバッファに格納し、一定間隔でまとめて更新・評価・保存する。

In [10]:
ac = Policy()                    # 学習するネットワーク (アクター + 双子 Q)
ac_targ = deepcopy(ac)           # ターゲット網 (polyak 平均でゆっくり追従)

ac.to(device)
ac_targ.to(device)

for p in ac_targ.parameters():   # ターゲット網は勾配では更新しない
    p.requires_grad = False

q_params = list(itertools.chain(ac.q1.parameters(), ac.q2.parameters()))
ac_params = list(ac.parameters())             # polyak 更新用 (_foreach_ に渡すリスト)
ac_targ_params = list(ac_targ.parameters())

if SNN_COMPILE:
    # compile 済みモジュールは deepcopy できないので、ac_targ を作った後に掛ける
    ac.san.forward = torch.compile(ac.san.forward, dynamic=False)
    ac_targ.san.forward = torch.compile(ac_targ.san.forward, dynamic=False)
    # compile 下では目標方策ノイズの乱数列が変わるので、SNN_COMPILE を切り替えると軌跡は再現しない
    compute_loss_q = torch.compile(compute_loss_q, dynamic=False)
    compute_loss_san = torch.compile(compute_loss_san, dynamic=False)
    print("actor と損失関数を torch.compile しました (形状ごとに初回のみコンパイルが走ります)")

san_optimizer = torch.optim.Adam(ac.san.parameters(), lr=san_lr, fused=True)
q_optimizer = torch.optim.Adam(q_params, lr=q_lr, fused=True)

replay_buffer = ReplayBuffer(
    obs_dim=obs_dim,
    act_dim=act_dim,
    mem_dim=mem_dim,
    size=replay_size,
    clip_limit=norm_clip_limit,
    device=device,
)

writer = SummaryWriter(logdir)
o = env.reset()
m = new_mem(NUM_ENVS)
ep_ret = torch.zeros(NUM_ENVS, device=device)
ep_len = torch.zeros(NUM_ENVS, device=device)
# ログ用の GPU 側アキュムレータ (毎反復で .item() すると同期が入るのでまとめて読む)
ret_sum = torch.zeros((), device=device)
ret_cnt = torch.zeros((), device=device)
len_sum = torch.zeros((), device=device)
loss_q_sum = torch.zeros((), device=device)
loss_san_sum = torch.zeros((), device=device)
parts_sum = torch.zeros(len(Go2ImitationWarpEnv.REWARD_KEYS), device=device)
fallen_sum = torch.zeros((), device=device)
n_q = n_san = n_parts = 0                     # 件数は python 側で数える (同期を避けるため)
t0 = time.time()

# 1 反復 = 全ワールド 1 制御ステップ (= NUM_ENVS 本の遷移) + updates_per_iter 回の勾配更新
for it in range(total_iters):
    # --- 行動を選ぶ ---
    a, m2 = get_action(replay_buffer.normalize_obs(o), m, act_noise)
    if it < start_iters:
        a = torch.rand_like(a) * 2 - 1                 # 初期はランダム行動で探索

    # --- 環境を進めてリプレイに格納する ---
    o2, r, d, info = env.step(a)
    ep_ret += r; ep_len += 1
    parts_sum += info["reward_parts"].mean(0); fallen_sum += info["fallen"].float().mean()
    n_parts += 1
    # 時間切れは done 扱いしない (転倒のみブートストラップを切る)
    d_store = (info["fallen"] & ~info["timeout"]).float()
    replay_buffer.store_batch(o, a, r, o2, d_store, m, m2)

    ret_sum += (ep_ret * d).sum(); len_sum += (ep_len * d).sum(); ret_cnt += d.sum()
    ep_ret = ep_ret * ~d; ep_len = ep_len * ~d

    o = env.autoreset(d)                               # 終了ワールドだけ参照姿勢へ
    m = torch.where(d.unsqueeze(1), new_mem(NUM_ENVS), m2)

    # --- 勾配更新 ---
    if it >= update_after_iters:
        for j in range(updates_per_iter):
            loss_q, loss_san = update(replay_buffer.sample_batch(batch_size), j)
            loss_q_sum += loss_q; n_q += 1
            if loss_san is not None:                   # actor は policy_delay 回に 1 回
                loss_san_sum += loss_san; n_san += 1

    # --- チェックポイントと動画 ---
    if (it + 1) % save_interval == 0:
        k = (it + 1) // 1000
        torch.save(ac.san.state_dict(), f"{model_dir}/model_imit_{k}Kit.pt")
        np.savez(f"{model_dir}/model_imit_mean_var_{k}Kit.npz",
                 mean=replay_buffer.mean.cpu().numpy(), var=replay_buffer.var.cpu().numpy())
        save_training_video(it + 1)

    # --- 固定指令での評価と TensorBoard ログ ---
    if (it + 1) % eval_interval == 0:
        res = test_agent()
        # 評価間隔内に 1 本もエピソードが終わらないことがあるので、その場合は次回に持ち越す
        n_ep = ret_cnt.item()
        avg_ret = ret_sum.item() / n_ep if n_ep else float('nan')
        avg_len = len_sum.item() / n_ep if n_ep else float('nan')
        if n_ep:
            ret_sum.zero_(); len_sum.zero_(); ret_cnt.zero_()
        steps = (it + 1) * NUM_ENVS
        writer.add_scalar("train/ep_ret", avg_ret, steps)
        writer.add_scalar("train/ep_len", avg_len, steps)
        for name, v in res.items():
            writer.add_scalar(f"test/{name}", v, steps)

        # 報酬の内訳 (1 ステップあたりの平均) と転倒率
        for tag, v in zip(Go2ImitationWarpEnv.REWARD_KEYS, (parts_sum / n_parts).tolist()):
            writer.add_scalar(tag, v, steps)
        writer.add_scalar("train/fallen_rate", fallen_sum.item() / n_parts, steps)
        parts_sum.zero_(); fallen_sum.zero_(); n_parts = 0

        # 損失 (total は別々の目的関数の和なので参考値)
        loss_c = loss_q_sum.item() / n_q if n_q else float('nan')
        loss_a = loss_san_sum.item() / n_san if n_san else float('nan')
        if n_q:
            writer.add_scalar("loss/critic", loss_c, steps)
        if n_san:
            writer.add_scalar("loss/actor", loss_a, steps)
            writer.add_scalar("loss/total", loss_c + loss_a, steps)
        loss_q_sum.zero_(); loss_san_sum.zero_(); n_q = n_san = 0

        print(f"{it+1:6d} it / {steps/1000:7.0f}K 遷移 ({time.time()-t0:.0f}s)  "
              f"直近{int(n_ep):5d}エピソード平均={avg_ret:7.1f} (長さ{avg_len:5.0f})  "
              f"損失 critic={loss_c:7.3f} actor={loss_a:7.3f}  評価: {res}")

writer.close()
print("学習完了")

if ENV_COLAB:
    runtime.unassign()


actor と損失関数を torch.compile しました (形状ごとに初回のみコンパイルが走ります)
リプレイバッファ: 100,000 遷移 × 52.3 KB = 4.99 GB (GPU)
  1000 it /     256K 遷移 (82s)  直近13570エピソード平均=   -0.7 (長さ   15)  損失 critic=  0.504 actor=  2.113  評価: {'前進': 0.7, '後退': 0.7, '左移動': 25.9, '左旋回': 23.0}
  2000 it /     512K 遷移 (155s)  直近  614エピソード平均=  100.6 (長さ  396)  損失 critic=  0.068 actor=  0.176  評価: {'前進': 109.4, '後退': 103.5, '左移動': 84.2, '左旋回': 251.8}
  3000 it /     768K 遷移 (208s)  直近  546エピソード平均=  186.3 (長さ  471)  損失 critic=  0.099 actor= -3.321  評価: {'前進': 162.2, '後退': 156.6, '左移動': 100.6, '左旋回': 303.9}
  4000 it /    1024K 遷移 (275s)  直近  544エピソード平均=  213.4 (長さ  468)  損失 critic=  0.146 actor= -6.305  評価: {'前進': 123.0, '後退': 161.5, '左移動': 91.0, '左旋回': 308.1}
  5000 it /    1280K 遷移 (329s)  直近  557エピソード平均=  214.2 (長さ  461)  損失 critic=  0.202 actor= -8.153  評価: {'前進': 145.9, '後退': 112.6, '左移動': 85.9, '左旋回': 303.7}
  6000 it /    1536K 遷移 (386s)  直近  535エピソード平均=  220.2 (長さ  475)  損失 critic=  0.256 actor= -9.168  評価: {'前進': 148.9, '後